In [1]:
import pandas as pd
import numpy as np
import yfinance as yf
import matplotlib.pyplot as plt

print("all imports successful")

all imports successful


testing the yfinance using apple's stock data from last month.

In [3]:
ticker = yf.Ticker("AAPL")
hist = ticker.history(period="1mo")
print(hist.head())

                                 Open        High         Low       Close  \
Date                                                                        
2026-03-11 00:00:00-04:00  261.089996  262.130005  259.549988  260.809998   
2026-03-12 00:00:00-04:00  258.660004  258.950012  254.179993  255.759995   
2026-03-13 00:00:00-04:00  255.479996  256.329987  249.520004  250.119995   
2026-03-16 00:00:00-04:00  252.110001  253.889999  249.880005  252.820007   
2026-03-17 00:00:00-04:00  252.960007  255.130005  252.179993  254.229996   

                             Volume  Dividends  Stock Splits  
Date                                                          
2026-03-11 00:00:00-04:00  26218900        0.0           0.0  
2026-03-12 00:00:00-04:00  40794000        0.0           0.0  
2026-03-13 00:00:00-04:00  36930000        0.0           0.0  
2026-03-16 00:00:00-04:00  32074200        0.0           0.0  
2026-03-17 00:00:00-04:00  32361600        0.0           0.0  


## step1: defining the stock universe and pulling earnings data

In [6]:
# analyzing stock of apple, microsoft, google, meta, amazon
TICKERS = ["AAPL", "MSFT", "GOOGL", "META", "AMZN"]

# earnings per share data for each company
earnings_dict = {}
for ticker_symbol in TICKERS:
    ticker = yf.Ticker(ticker_symbol)
    earnings = ticker.earnings_dates
    earnings_dict[ticker_symbol] = earnings
    print(f"{ticker_symbol}: found {len(earnings)} earnings dates")

print(earnings_dict["AAPL"]) # looking at apple's data up-close

AAPL: found 25 earnings dates
MSFT: found 25 earnings dates
GOOGL: found 25 earnings dates
META: found 25 earnings dates
AMZN: found 25 earnings dates
                           EPS Estimate  Reported EPS  Surprise(%)
Earnings Date                                                     
2026-04-30 16:00:00-04:00          1.94           NaN          NaN
2026-01-29 16:00:00-05:00          2.67          2.84         6.25
2025-10-30 16:00:00-04:00          1.77          1.85         4.52
2025-07-31 16:00:00-04:00          1.43          1.57         9.48
2025-05-01 16:00:00-04:00          1.63          1.65         1.50
2025-01-30 16:00:00-05:00          2.35          2.40         2.26
2024-10-31 16:00:00-04:00          1.60          1.64         2.28
2024-08-01 16:00:00-04:00          1.34          1.40         4.30
2024-05-02 16:00:00-04:00          1.51          1.53         1.46
2024-02-01 16:00:00-05:00          2.10          2.18         3.66
2023-11-02 16:00:00-04:00          1.39      

In [8]:
# saving raw data in data folder
import os
for ticker_symbol, df in earnings_dict.items():
    if df is not None:
        filepath = f"../data/{ticker_symbol}_earnings.csv"
        df.to_csv(filepath)
        print(f"Saved {ticker_symbol} to {filepath}")

Saved AAPL to ../data/AAPL_earnings.csv
Saved MSFT to ../data/MSFT_earnings.csv
Saved GOOGL to ../data/GOOGL_earnings.csv
Saved META to ../data/META_earnings.csv
Saved AMZN to ../data/AMZN_earnings.csv


In [9]:
# pulling stock price hsitory for all tickets
prices_dict = {}

for ticker_symbol in TICKERS:
    ticker = yf.Ticker(ticker_symbol)
    hist = ticker.history(period="2y")  # 2 years of daily prices
    prices_dict[ticker_symbol] = hist
    print(f"{ticker_symbol}: {len(hist)} days of price data")

AAPL: 501 days of price data
MSFT: 501 days of price data
GOOGL: 501 days of price data
META: 501 days of price data
AMZN: 501 days of price data


In [10]:
# saving price data too
for ticker_symbol, df in prices_dict.items():
    filepath = f"../data/{ticker_symbol}_prices.csv"
    df.to_csv(filepath)
    print(f"Saved {ticker_symbol} prices to {filepath}")

Saved AAPL prices to ../data/AAPL_prices.csv
Saved MSFT prices to ../data/MSFT_prices.csv
Saved GOOGL prices to ../data/GOOGL_prices.csv
Saved META prices to ../data/META_prices.csv
Saved AMZN prices to ../data/AMZN_prices.csv


## step2: download earnings call transcripts from SEC EDGAR 

In [11]:
from sec_edgar_downloader import Downloader
# initializing the downloader
dl = Downloader("Hansini Rajesh", "hansinirajesh92@gmail.com", "../data/sec_filings")


In [14]:
# downloading 8-K filngs for all tickets
for ticker_symbol in TICKERS:
    print(f"downloading filings for {ticker_symbol}...")
    dl.get("8-K", ticker_symbol, limit=10)  # last 10 filings per company
    print(f"done with {ticker_symbol}")

downloading filings for AAPL...
done with AAPL
downloading filings for MSFT...
done with MSFT
downloading filings for GOOGL...
done with GOOGL
downloading filings for META...
done with META
downloading filings for AMZN...
done with AMZN


In [15]:
import os
sec_path = "../data/sec_filings"
for ticker_symbol in TICKERS:
    ticker_path = os.path.join(sec_path, "sec-edgar-filings", ticker_symbol, "8-K")
    if os.path.exists(ticker_path):
        num_filings = len(os.listdir(ticker_path))
        print(f"{ticker_symbol}: {num_filings} filings downloaded")
    else:
        print(f"{ticker_symbol}: folder not found")

AAPL: 10 filings downloaded
MSFT: 10 filings downloaded
GOOGL: 10 filings downloaded
META: 10 filings downloaded
AMZN: 10 filings downloaded


In [17]:
# checking the filings for APPLE
aapl_path = os.path.join(sec_path, "sec-edgar-filings", "AAPL", "8-K")
first_filing = os.listdir(aapl_path)[0]
filing_folder = os.path.join(aapl_path, first_filing)

print(f"filing folder: {first_filing}")
print(f"files inside:")
for f in os.listdir(filing_folder):
    print(f"  - {f}")

filing folder: 0001140361-25-018400
files inside:
  - full-submission.txt


In [18]:
# reading raw text of the first AAPL filing
filing_path = os.path.join(filing_folder, "full-submission.txt")

with open(filing_path, "r", encoding="utf-8", errors="ignore") as f:
    raw_text = f.read()

# print first 2000 characters
print(raw_text[:2000])

<SEC-DOCUMENT>0001140361-25-018400.txt : 20250512
<SEC-HEADER>0001140361-25-018400.hdr.sgml : 20250512
<ACCEPTANCE-DATETIME>20250512163028
ACCESSION NUMBER:		0001140361-25-018400
CONFORMED SUBMISSION TYPE:	8-K
PUBLIC DOCUMENT COUNT:		20
CONFORMED PERIOD OF REPORT:	20250505
ITEM INFORMATION:		Other Events
ITEM INFORMATION:		Financial Statements and Exhibits
FILED AS OF DATE:		20250512
DATE AS OF CHANGE:		20250512

FILER:

	COMPANY DATA:	
		COMPANY CONFORMED NAME:			Apple Inc.
		CENTRAL INDEX KEY:			0000320193
		STANDARD INDUSTRIAL CLASSIFICATION:	ELECTRONIC COMPUTERS [3571]
		ORGANIZATION NAME:           	06 Technology
		EIN:				942404110
		STATE OF INCORPORATION:			CA
		FISCAL YEAR END:			0927

	FILING VALUES:
		FORM TYPE:		8-K
		SEC ACT:		1934 Act
		SEC FILE NUMBER:	001-36743
		FILM NUMBER:		25935352

	BUSINESS ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		CITY:			CUPERTINO
		STATE:			CA
		ZIP:			95014
		BUSINESS PHONE:		(408) 996-1010

	MAIL ADDRESS:	
		STREET 1:		ONE APPLE PARK WAY
		

In [19]:
print(f"total characters in this filing: {len(raw_text):,}")
print(f"total words (approx): {len(raw_text.split()):,}")

total characters in this filing: 887,109
total words (approx): 71,013


In [20]:
# keywords that appear in actual earnings call transcripts
transcript_keywords = ["earnings call", "conference call", "operator", "questions and answers", "Q&A"]

for keyword in transcript_keywords:
    if keyword.lower() in raw_text.lower():
        print(f"found: '{keyword}'")
    else:
        print(f"NOT found: '{keyword}'")

NOT found: 'earnings call'
NOT found: 'conference call'
NOT found: 'operator'
NOT found: 'questions and answers'
NOT found: 'Q&A'


In [21]:
# checking all 10 APPLE filings for transcript content
aapl_filings = os.listdir(aapl_path)

for filing in aapl_filings:
    path = os.path.join(aapl_path, filing, "full-submission.txt")
    with open(path, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()
    
    has_transcript = any(kw.lower() in text.lower() for kw in ["conference call", "operator", "earnings call"])
    print(f"{filing}: {'HAS TRANSCRIPT' if has_transcript else 'press release only'}")

0001140361-25-018400: press release only
0000320193-25-000077: HAS TRANSCRIPT
0000320193-25-000071: HAS TRANSCRIPT
0001140361-25-027340: press release only
0001140361-26-006577: press release only
0000320193-25-000055: HAS TRANSCRIPT
0000320193-26-000005: HAS TRANSCRIPT
0001140361-25-025275: press release only
0001140361-26-000199: press release only
0001140361-25-044561: press release only


In [22]:
# accross all tickets, checking for transcript content
transcript_counts = {}

for ticker_symbol in TICKERS:
    ticker_path = os.path.join(sec_path, "sec-edgar-filings", ticker_symbol, "8-K")
    filings = os.listdir(ticker_path)
    count = 0
    
    for filing in filings:
        path = os.path.join(ticker_path, filing, "full-submission.txt")
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        if any(kw.lower() in text.lower() for kw in ["conference call", "operator"]):
            count += 1
    
    transcript_counts[ticker_symbol] = count
    print(f"{ticker_symbol}: {count} out of 10 filings contain transcripts")

AAPL: 4 out of 10 filings contain transcripts
MSFT: 5 out of 10 filings contain transcripts
GOOGL: 3 out of 10 filings contain transcripts
META: 4 out of 10 filings contain transcripts
AMZN: 4 out of 10 filings contain transcripts


In [32]:
from bs4 import BeautifulSoup
import re

def extract_transcript_text(filepath):
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        raw = f.read()
    try:
        # parsing out HTML sections using BeautifulSoup
        soup = BeautifulSoup(raw, "html.parser")
        text = soup.get_text(separator=" ")
    except Exception:
        # if parser fails, just use raw text
        text = raw
    
    # cleaning up extra whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text

In [33]:
transcript_files = []

for ticker_symbol in TICKERS:
    ticker_path = os.path.join(sec_path, "sec-edgar-filings", ticker_symbol, "8-K")
    filings = os.listdir(ticker_path)
    
    for filing in filings:
        path = os.path.join(ticker_path, filing, "full-submission.txt")
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            text = f.read()
        
        if any(kw.lower() in text.lower() for kw in ["conference call", "operator"]):
            transcript_files.append({
                "ticker": ticker_symbol,
                "filing_id": filing,
                "filepath": path
            })

print(f"total no. of transcripts found: {len(transcript_files)}")
for t in transcript_files:
    print(f"  {t['ticker']} - {t['filing_id']}")

total no. of transcripts found: 20
  AAPL - 0000320193-25-000077
  AAPL - 0000320193-25-000071
  AAPL - 0000320193-25-000055
  AAPL - 0000320193-26-000005
  MSFT - 0000950170-25-010484
  MSFT - 0001193125-25-256310
  MSFT - 0001193125-26-027198
  MSFT - 0000950170-25-100226
  MSFT - 0000950170-25-061032
  GOOGL - 0001652044-26-000012
  GOOGL - 0001652044-25-000087
  GOOGL - 0001652044-25-000056
  META - 0001628280-26-003832
  META - 0001326801-25-000050
  META - 0001628280-25-036719
  META - 0001628280-25-047114
  AMZN - 0001018724-26-000002
  AMZN - 0001018724-25-000121
  AMZN - 0001018724-25-000084
  AMZN - 0001018724-25-000034


In [34]:
# extracting clean text from all the transcripts
for item in transcript_files:
    clean_text = extract_transcript_text(item["filepath"])
    item["full_text"] = clean_text
    print(f"{item['ticker']} - {item['filing_id']}: {len(clean_text.split())} words")

AAPL - 0000320193-25-000077: 8775 words
AAPL - 0000320193-25-000071: 7837 words
AAPL - 0000320193-25-000055: 18471 words
AAPL - 0000320193-26-000005: 7605 words
MSFT - 0000950170-25-010484: 9055 words
MSFT - 0001193125-25-256310: 673847 words
MSFT - 0001193125-26-027198: 9246 words
MSFT - 0000950170-25-100226: 8718 words
MSFT - 0000950170-25-061032: 9167 words
GOOGL - 0001652044-26-000012: 12469 words
GOOGL - 0001652044-25-000087: 10864 words
GOOGL - 0001652044-25-000056: 10393 words
META - 0001628280-26-003832: 9971 words
META - 0001326801-25-000050: 9662 words
META - 0001628280-25-036719: 9827 words
META - 0001628280-25-047114: 10253 words
AMZN - 0001018724-26-000002: 18367 words
AMZN - 0001018724-25-000121: 18048 words
AMZN - 0001018724-25-000084: 17436 words
AMZN - 0001018724-25-000034: 17485 words


In [35]:
# splitting trascripts to ceo and q&a sections
def split_transcript_sections(text):
    text_lower = text.lower()
    
    # finding where q&a starts
    qa_markers = ["question-and-answer", "questions and answers", "q&a session", "open the line"]
    qa_start = None
    
    for marker in qa_markers:
        idx = text_lower.find(marker)
        if idx != -1:
            qa_start = idx
            break
    
    if qa_start:
        prepared_remarks = text[:qa_start]
        qa_section = text[qa_start:]
    else:
        prepared_remarks = text
        qa_section = ""
    
    return prepared_remarks, qa_section

# test on first transcript
remarks, qa = split_transcript_sections(transcript_files[0]["full_text"])
print(f"prepared remarks: {len(remarks.split())} words")
print(f"q&a section: {len(qa.split())} words")

prepared remarks: 8775 words
q&a section: 0 words


In [36]:
# applying split to all transcripts and saving to json
import json

for item in transcript_files:
    remarks, qa = split_transcript_sections(item["full_text"])
    item["prepared_remarks"] = remarks
    item["qa_section"] = qa
    # removing full_text to keep file size small
    del item["full_text"]

# saveing to json
output_path = "../data/transcripts_clean.json"
with open(output_path, "w") as f:
    json.dump(transcript_files, f, indent=2)

print(f"saved {len(transcript_files)} transcripts to {output_path}")

saved 20 transcripts to ../data/transcripts_clean.json
